# Task 3 ? Local model benchmarking

This notebook is the orchestration and review layer. Model loading, inference measurement, aggregation, and artifact writing live in the `task3_benchmark` Python package. Run the FP16 and 4-bit cells on a CUDA GPU to refresh the results.


In [ ]:
import sys
from pathlib import Path
import torch
from IPython.display import display

# Locate this Task 3 directory whether Jupyter starts here or at the repo root.
working_dir = Path.cwd()
if working_dir.name == "optimization(Task_3)":
    TASK3_DIR = working_dir
elif (working_dir / "optimization(Task_3)").is_dir():
    TASK3_DIR = working_dir / "optimization(Task_3)"
else:
    TASK3_DIR = working_dir
if str(TASK3_DIR) not in sys.path:
    sys.path.insert(0, str(TASK3_DIR))

from task3_benchmark import (
    MODEL_NAME, OUTPUT_DIR, TEST_PROMPTS, load_model, clear_cuda_cache,
    benchmark_model, compare_summaries, save_benchmark_artifacts,
)

print(f"Model: {MODEL_NAME}")
print(f"Prompts: {len(TEST_PROMPTS)}")
print(f"Results directory: {OUTPUT_DIR}")


## Run the FP16 configuration

This measures each prompt's generation time and peak CUDA allocated and reserved memory, then saves the per-prompt CSV.


In [ ]:
fp16_model, fp16_tokenizer = load_model("FP16")
fp16_results = benchmark_model(fp16_model, fp16_tokenizer, "FP16")
fp16_results.to_csv(OUTPUT_DIR / "fp16_results.csv", index=False)
display(fp16_results[[
    "Prompt_ID", "Generated_Tokens", "Generation_Time_Seconds",
    "Tokens_Per_Second", "Peak_GPU_Allocated_GB", "Peak_GPU_Reserved_GB"
]])
del fp16_model, fp16_tokenizer
clear_cuda_cache()


## Run the 4-bit NF4 configuration


In [ ]:
int4_model, int4_tokenizer = load_model("4BIT")
int4_results = benchmark_model(int4_model, int4_tokenizer, "4BIT")
int4_results.to_csv(OUTPUT_DIR / "4bit_results.csv", index=False)
display(int4_results[[
    "Prompt_ID", "Generated_Tokens", "Generation_Time_Seconds",
    "Tokens_Per_Second", "Peak_GPU_Allocated_GB", "Peak_GPU_Reserved_GB"
]])
del int4_model, int4_tokenizer
clear_cuda_cache()


## Compare and write submission artifacts

This creates the performance summary, paired response CSV, quality scoring worksheet, charts, and submission report in `results/`. Existing human quality scores are preserved.


In [ ]:
comparison, memory_saving, speed_difference = compare_summaries(
    fp16_results, int4_results
)
display(comparison)
print(f"4-bit peak allocated memory reduction: {memory_saving:.2f}%")
print(f"4-bit speed change vs FP16: {speed_difference:.2f}%")

quality_comparison, quality_scores, submission = save_benchmark_artifacts(
    fp16_results=fp16_results,
    int4_results=int4_results,
    summary=comparison,
    memory_saving=memory_saving,
    speed_difference=speed_difference,
    gpu_name=torch.cuda.get_device_name(0),
    output_dir=OUTPUT_DIR,
)
display(quality_comparison)
print(f"Submission report saved to {OUTPUT_DIR / 'Task_3_Submission.md'}")


## Quality review

Compare the FP16 and 4-bit response for each prompt in `results/quality_comparison.csv`. Rate correctness, relevance, and clarity from 1 (poor) to 5 (strong) in `results/quality_scores_template.csv`, and write a concrete observation for each pair. Rerun the final artifact cell afterward to update the submission report.

The report includes the benchmark measurements, a KV cache explanation, why quantization reduces weight memory, and a stated 24 GB / 20-user serving plan. The quality section remains marked pending until the ratings and observations are filled in.
